# Notebook: Store Sales Time Series Forecasting
Romain Sebire - 125 009 460

**Objective:** Predict daily sales for stores in the Corporación Favorita grocery chain, using time series features and external data sources (holidays, oil prices, paydays).

**Dataset:** [Store Sales — Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting) from Kaggle, with sales data for 54 stores and 33 product families.

**Approach:** Iterative feature enrichment — starting with basic temporal features, then adding holiday, oil price, and payday information. Models: RandomForest and LightGBM.

**Key Result:** Public score of **0.69857** on Kaggle using LightGBM with all enriched features.

## Table of Contents

1. [Data Loading](#Data-Loading)
2. [Feature Overview](#Feature-Overview)
3. [Baseline Model: Store-Level Sales Prediction](#Baseline-Model:-Store-Level-Sales-Prediction)
    - [Feature Engineering](#Feature-Engineering)
    - [Exploratory Data Analysis](#Exploratory-Data-Analysis)
    - [Missing Value Treatment](#Missing-Value-Treatment)
    - [Model Comparison](#Model-Comparison)
4. [Adding Holiday Features](#Adding-Holiday-Features)
5. [Adding Oil Price Features](#Adding-Oil-Price-Features)
6. [Adding Payday Features](#Adding-Payday-Features)
7. [Final Prediction: All Stores and Products](#Final-Prediction:-All-Stores-and-Products)
8. [Kaggle Results](#Kaggle-Results)

## Data Loading

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

train = pd.read_csv('train.csv', parse_dates=['date'])
test = pd.read_csv('test.csv', parse_dates=['date'])
oil = pd.read_csv('oil.csv', parse_dates=['date'])
holidays = pd.read_csv('holidays_events.csv', parse_dates=['date'])
stores = pd.read_csv('stores.csv')
transactions = pd.read_csv('transactions.csv', parse_dates=['date'])

In [ ]:
# Definition of the RMSLE evaluation method
def rmsle(y_true, y_pred):
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))

### Display of Variables (Features) and Objects of Each DataFrame

In [ ]:
import pandas as pd

# Dicionário de todos os DataFrames
datasets = {
    "train": train,
    "oil": oil,
    "holidays": holidays,
    "stores": stores,
    "transactions": transactions
}

# Percorra cada DataFrame
for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    print("Colonnes :", df.columns.tolist())
    for col in df.select_dtypes(include='object').columns:
        print(f"\n - {col} ({df[col].nunique()} objets) :")
        print(df[col].unique())


## Baseline Model: Store-Level Sales Prediction

In [ ]:
# Concatenate train and test DataFrames to apply the same transformations

train['is_test'] = 0
test['is_test'] = 1

df = pd.concat([train, test], ignore_index=True)

### Feature Engineering

In [ ]:
df["day_of_week"] = df["date"].dt.dayofweek
df['day_of_year'] = df['date'].dt.dayofyear
df["day_of_month"] = df["date"].dt.day
df["month"] = df["date"].dt.month
df["year"] = df["date"].dt.year
df['rolling_mean_30'] = df.groupby(['store_nbr','family'])['sales'].transform(lambda x: x.rolling(30, min_periods=1).mean())


In [ ]:
df = df.merge(
    stores[['store_nbr', 'city', 'state', 'type', 'cluster']],
    on='store_nbr',
    how='left'
)

df = df.merge(
    transactions[['store_nbr', 'date', 'transactions']],
    on=['store_nbr', 'date'],
    how='left'
)

### Exploratory Data Analysis (EDA)

#### Distribution Analysis

In [ ]:
df_analysis = df[df['is_test'] == 0].drop(columns='is_test')
cat_vars = df_analysis.select_dtypes(include='object')

plt.figure(figsize=(10, 12))
for i, col in enumerate(cat_vars, 1):
    plt.subplot(2, 2, i)

    top_categories = df_analysis[col].value_counts().nlargest(10)
    top_cat_names = top_categories.index
    data = df_analysis[df_analysis[col].isin(top_cat_names)]
    palette = sns.color_palette("husl", len(top_cat_names))
    color_dict = dict(zip(top_cat_names, palette))

    sns.countplot(
        data=data,
        y=col,
        order=top_cat_names,
        hue=col,
        palette=color_dict,
        dodge=False,
        legend=False
    )

    plt.title(f'Distribution of {col}')
    plt.tight_layout()

plt.show()



In [ ]:
# Convert float16 to float32 to avoid NotImplementedError
df_analysis['onpromotion'] = df_analysis['onpromotion'].astype('float32')

# Define numerical variables
num_vars = ['sales', 'onpromotion', 'transactions']

# Plot distributions
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_vars, 1):
    plt.subplot(2, 2, i)
    
    # Filter out NaN values for the current column
    plot_data = df_analysis[col].dropna()
    
    # Use try-except to handle potential plotting issues
    try:
        sns.histplot(data=plot_data, kde=True, bins=50)
        plt.title(f'Distribution of {col}')
    except Exception as e:
        print(f"Could not plot {col}: {str(e)}")
        
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12, 6))
sns.lineplot(data=df_analysis.groupby('date')['sales'].sum().reset_index(), x='date', y='sales')
plt.title('Daily Sales Trend')
plt.show()

#### Visualization of Relationships with the Target Variable (Sales)

In [ ]:
data = (
    df_analysis[["type", "sales"]]
    .groupby("type", as_index=False)
    .mean()
    .sort_values(by="sales", ascending=False)
)

# Barplot
plt.figure(figsize=(8, 5))
sns.barplot(data=data, x="type", y="sales", hue="type", palette="viridis", legend=False)

plt.title("Average Sales by Store Type")
plt.xlabel("Tipo de loja")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.show()


In [ ]:
data = df_analysis[["family", "sales"]].groupby(["family"], as_index=False).mean().sort_values(by="sales", ascending=False).head(10)

# Barplot
plt.figure(figsize=(8, 5))
sns.barplot(data=data, x="family", y="sales", hue="family", palette="viridis", legend=False)

plt.title("Average Sales por família")
plt.xlabel("Product Family")
plt.ylabel("Average Sales")
plt.tight_layout()
plt.xticks(rotation=45)
plt.show()

In [ ]:
plt.figure(figsize=(10,6))
sns.boxplot(x=df_analysis["onpromotion"] > 0, 
            y=np.log1p(df["sales"]))
plt.title("Promotion Impact on Sales")
plt.xlabel("Item on Promotion")
plt.ylabel("Log(Sales + 1)")
plt.xticks([0,1], ["No Promotion", "Promoted"])
plt.show()


In [ ]:
monthly_sales = df_analysis.groupby(["year", "month"])["sales"].sum().reset_index()
plt.figure(figsize=(12,6))
sns.lineplot(x="month", y="sales", hue="year", data=monthly_sales, marker="o")
plt.title("Monthly Sales Trends by Year")
plt.xticks(range(1,13))
plt.show()

In [ ]:
print(df)

In [ ]:
# Map day numbers to day names
day_name_map = {
    0: "Monday", 1: "Tuesday", 2: "Wednesday", 3: "Thursday",
    4: "Friday", 5: "Saturday", 6: "Sunday"
}
# Substituir os números pelos nomes
dow_sales = df_analysis.groupby("day_of_week")["sales"].mean().reset_index()
dow_sales["day_name"] = dow_sales["day_of_week"].map(day_name_map)
sns.barplot(data=dow_sales, x="day_name", y="sales", hue="day_name", palette="viridis", legend=False)
plt.title("Average Sales by Day of Week")
plt.xticks(rotation=45)
plt.show()

### Missing Value Search and Replacement

In [ ]:
# Summary of missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100)

missing_summary = pd.concat([missing, missing_pct], axis=1)
missing_summary.columns = ['Total Missing', 'Percentage (%)']
missing_summary = missing_summary[missing_summary['Total Missing'] > 0]
missing_summary = missing_summary.sort_values(by='Percentage (%)', ascending=False)

print(missing_summary)


Missing values in "sales" correspond to the data to be predicted.

9% of values are missing in the transactions table; we will replace these missing values with the mean number of transactions for each store.

In [ ]:
store_trans_avg = df.groupby('store_nbr')['transactions'].mean().reset_index()
store_trans_avg.columns = ['store_nbr', 'store_mean_transactions']
df = df.merge(store_trans_avg, on='store_nbr', how='left')
df['transactions'] = df['transactions'].fillna(df['store_mean_transactions'])

### Model Comparison

RandomForestRegressor model: RMSLE = 0.4956

In [ ]:
store_id = 1
store_data = df[df['store_nbr'] == store_id].copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30', 
            'city', 'state', 'type', 'cluster', 'transactions']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    store_data[col] = store_data[col].astype('category').cat.codes

store_train_data = store_data[store_data['is_test'] == 0].drop(columns='is_test')

# Sort data by date
store_train_data = store_train_data.sort_values("date")

# Divisão manual
split_index = int(len(store_train_data) * 0.8)
train_data = store_train_data.iloc[:split_index]
val_data = store_train_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data["sales"]
X_val = val_data[features]
y_val = val_data["sales"]

# Treinar RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Previsão
y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)
score = rmsle(y_val, y_pred)
print(f"RMSLE: {score:.4f}")


LightGBM model: RMSLE = 0.5674

In [ ]:
store_id = 1
store_data = df[df['store_nbr'] == store_id].copy()

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30', 
            'city', 'state', 'type', 'cluster', 'transactions']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    store_data[col] = store_data[col].astype('category')

store_train_data = store_data[store_data['is_test'] == 0].drop(columns='is_test')

# Sort data by date
store_train_data = store_train_data.sort_values("date")

# Divisão manual
split_index = int(len(store_train_data) * 0.8)
train_data = store_train_data.iloc[:split_index]
val_data = store_train_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data["sales"]
X_val = val_data[features]
y_val = val_data["sales"]

# Treinar LightGBM
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=100,
    force_col_wise=True,
    random_state=42
)
model.fit(X_train, y_train)

# Previsão & RMSLE
y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)
score = rmsle(y_val, y_pred)
print(f"RMSLE: {score:.4f}")


Going forward, we will use the RandomForestRegressor model, which is more efficient.

## Adding Holiday Features

### Holiday Data Exploration

In [ ]:
df_analysis = df[df['is_test'] == 0].drop(columns='is_test')
cat_vars = holidays.select_dtypes(include='object')

plt.figure(figsize=(10, 12))
for i, col in enumerate(cat_vars, 1):
    plt.subplot(2, 2, i)

    top_categories = holidays[col].value_counts().nlargest(10)
    top_cat_names = top_categories.index
    data = holidays[holidays[col].isin(top_cat_names)]
    palette = sns.color_palette("husl", len(top_cat_names))
    color_dict = dict(zip(top_cat_names, palette))

    sns.countplot(
        data=data,
        y=col,
        order=top_cat_names,
        hue=col,
        palette=color_dict,
        dodge=False,
        legend=False
    )

    plt.title(f'Distribution of {col}')
    plt.tight_layout()

plt.show()

### Holiday Feature Integration

In [ ]:
# Filter non-transferred holidays
holidays_filtered = holidays[(holidays['transferred'] == False) & (holidays['type'] != 'Work_day')]

# Feriados nacionais (válidos em todo lugar)
national_holidays = holidays_filtered[holidays_filtered['locale'] == 'National'][['date']].drop_duplicates()
national_holidays['is_national'] = 1

# Feriados regionais (por estado)
regional_holidays = holidays_filtered[holidays_filtered['locale'] == 'Regional'][['date', 'locale_name']].drop_duplicates()
regional_holidays['is_regional'] = 1
regional_holidays = regional_holidays.rename(columns={'locale_name': 'state'})

# Feriados locais (por cidade)
local_holidays = holidays_filtered[holidays_filtered['locale'] == 'Local'][['date', 'locale_name']].drop_duplicates()
local_holidays['is_local'] = 1
local_holidays = local_holidays.rename(columns={'locale_name': 'city'})

# Fusão dos 3 tipos de feriados no df
df = df.merge(national_holidays, on='date', how='left')
df = df.merge(regional_holidays, on=['date', 'state'], how='left')
df = df.merge(local_holidays, on=['date', 'city'], how='left')

# Fill NaN with 0 and create the final is_holiday
df['is_national'] = df['is_national'].fillna(0)
df['is_regional'] = df['is_regional'].fillna(0)
df['is_local'] = df['is_local'].fillna(0)

df['is_holiday'] = ((df['is_national'] + df['is_regional'] + df['is_local']) > 0).astype(int)

df['holiday_type'] = (
    df['is_national'].astype(int).map({1: 'National'})
    .fillna(df['is_regional'].astype(int).map({1: 'Regional'}))
    .fillna(df['is_local'].astype(int).map({1: 'Local'}))
    .fillna('None')
)

df['holiday_type'] = df['holiday_type'].astype('category').cat.codes



### Model with Holiday Features

In [ ]:
store_id = 1
store_data = df[df['store_nbr'] == store_id].copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30',
             'city', 'state', 'type', 'cluster', 'transactions', 'holiday_type']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    df[col] = df[col].astype('category').cat.codes

store_train_data = store_data[store_data['is_test'] == 0].drop(columns='is_test')

# Sort data by date
store_train_data = store_train_data.sort_values("date")

# Divisão manual
split_index = int(len(store_train_data) * 0.8)
train_data = store_train_data.iloc[:split_index]
val_data = store_train_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data["sales"]
X_val = val_data[features]
y_val = val_data["sales"]

# Treinar RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Previsão & RMSLE
y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)
score = rmsle(y_val, y_pred)
print(f"RMSLE: {score:.4f}")

RMSLE: 0.4931

## Adding Oil Price Features

In [ ]:
start = train.date.min().date()
end = test.date.max().date()

# Create the complete date series between start and end
full_dates = pd.date_range(start, end)

# Identify missing dates in the oil data
missing_oil_dates = full_dates.difference(oil['date'])

# Number and percentage of missing dates
num_missing = len(missing_oil_dates)
total_dates = len(full_dates)
percent_missing = 100 * num_missing / total_dates

print("Análise dos dados de petróleo ausentes:")
print(f"- Total de datas esperadas : {total_dates}")
print(f"- Datas ausentes           : {num_missing}")
print(f"- Porcentagem ausente      : {percent_missing:.2f}%")


In [ ]:
# Reconstruct a complete series with missing dates
oil_full = pd.DataFrame({"date": full_dates})
oil = oil_full.merge(oil, on="date", how="left").sort_values("date")


# Fusão com o dataframe principal
df = df.merge(oil, on="date", how="left")

In [ ]:
# Linear interpolation
df['dcoilwtico'] = df['dcoilwtico'].interpolate(method='linear', limit_direction='both')

In [ ]:
store_id = 1
store_data = df[df['store_nbr'] == store_id].copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30',
             'city', 'state', 'type', 'cluster', 'transactions', 'holiday_type', 'dcoilwtico']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    df[col] = df[col].astype('category').cat.codes

store_train_data = store_data[store_data['is_test'] == 0].drop(columns='is_test')

# Sort data by date
store_train_data = store_train_data.sort_values("date")

# Divisão manual
split_index = int(len(store_train_data) * 0.8)
train_data = store_train_data.iloc[:split_index]
val_data = store_train_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data["sales"]
X_val = val_data[features]
y_val = val_data["sales"]

# Treinar RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Previsão & RMSLE
y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)
score = rmsle(y_val, y_pred)
print(f"RMSLE: {score:.4f}")

RMSLE: 0.4737

## Adding Payday Features

In [ ]:
# Add payday period indicators
df["is_payday"] = ((df.date.dt.day == 15) | 
                     (df.date.dt.is_month_end)).astype(int)

# Add indicators for days after payday
df["is_day_after_payday"] = ((df.date.dt.day == 16) | 
                              (df.date.dt.day == 1)).astype(int)

df["is_two_days_after_payday"] = ((df.date.dt.day == 17) | 
                                   (df.date.dt.day == 2)).astype(int)

# Add payday week indicator (the entire week after payday may show increased sales)
df["is_payday_week"] = ((df.date.dt.day >= 15) & (df.date.dt.day <= 21) | 
                          (df.date.dt.day >= 1) & (df.date.dt.day <= 7)).astype(int)

In [ ]:
store_id = 1
store_data = df[df['store_nbr'] == store_id].copy()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30',
             'city', 'state', 'type', 'cluster', 'transactions', 'holiday_type', 'dcoilwtico',
             'is_payday', 'is_day_after_payday', 'is_two_days_after_payday', 'is_payday_week']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    df[col] = df[col].astype('category').cat.codes

store_train_data = store_data[store_data['is_test'] == 0].drop(columns='is_test')

# Sort data by date
store_train_data = store_train_data.sort_values("date")

# Divisão manual
split_index = int(len(store_train_data) * 0.8)
train_data = store_train_data.iloc[:split_index]
val_data = store_train_data.iloc[split_index:]

X_train = train_data[features]
y_train = train_data["sales"]
X_val = val_data[features]
y_val = val_data["sales"]

# Treinar RandomForestRegressor
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

# Previsão & RMSLE
y_pred = model.predict(X_val)
y_pred = np.maximum(0, y_pred)
score = rmsle(y_val, y_pred)
print(f"RMSLE: {score:.4f}")

RMSLE: 0.4722

## Final Prediction: All Stores and Products

In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import train_test_split

# # Features
# features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30',
#              'city', 'state', 'type', 'cluster', 'transactions', 'holiday_type', 'dcoilwtico',
#              'is_payday', 'is_day_after_payday', 'is_two_days_after_payday', 'is_payday_week']

# # Codificação das variáveis categóricas
# for col in ['family', 'city', 'state', 'type']:
#     df[col] = df[col].astype('category').cat.codes

# df_train = df[df['is_test'] == 0].drop(columns='is_test')

# # Sort data by date
# df_train = df_train.sort_values("date")

# X_train = df_train[features]
# y_train = df_train["sales"]

# # Treinar RandomForestRegressor
# model = RandomForestRegressor(n_estimators=100, random_state=42)
# model.fit(X_train, y_train)

The RandomForestRegressor model is too costly and slow; after several attempts, it generates an out-of-memory error. Therefore, we switch back to LGBMRegressor, which is slightly less performant but more resource-efficient.

In [ ]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# Features
features = ['family', 'onpromotion', 'day_of_week', 'day_of_year', 'month', 'year','rolling_mean_30',
             'city', 'state', 'type', 'cluster', 'transactions', 'holiday_type', 'dcoilwtico',
             'is_payday', 'is_day_after_payday', 'is_two_days_after_payday', 'is_payday_week']

# Codificação das variáveis categóricas
for col in ['family', 'city', 'state', 'type']:
    df[col] = df[col].astype('category')

df_train = df[df['is_test'] == 0].drop(columns='is_test')

# Sort data by date
df_train = df_train.sort_values("date")

X_train = df_train[features]
y_train = df_train["sales"]

# Treinar LightGBM
model = lgb.LGBMRegressor(
    objective='regression',
    n_estimators=100,
    force_col_wise=True,
    random_state=42
)
model.fit(X_train, y_train)

In [ ]:
df_test = df[df['is_test'] == 1].drop(columns='is_test')

# Previsão
X_test = df_test[features]
y_pred = model.predict(X_test)

# Force positive values
y_pred = np.maximum(0, y_pred)

df_test['sales'] = y_pred

# Prepare the submission
df_submission = df_test[['id', 'sales']]
print(df_submission.head())

submission_file_path = 'submission.csv'
df_submission.to_csv(submission_file_path, index=False)


## Kaggle Results

Public score: 0.69857

## Key Results

| Stage | Feature Added | RMSLE |
|-------|--------------|-------|
| Baseline (single store) | Temporal features only | 0.4956 |
| + Holidays | Holiday type indicators | 0.4931 |
| + Oil prices | Interpolated oil price data | 0.4737 |
| + Paydays | Payday period indicators | 0.4722 |
| **Final (all stores)** | **All features, LightGBM** | **0.69857** (Kaggle) |

**Key insight:** Each additional data source incrementally improved the model. The jump in RMSLE from single-store to all-stores prediction is expected due to the increased complexity of predicting across 54 stores × 33 product families simultaneously.